<a href="https://colab.research.google.com/github/antonytfchoi-glitch/capstone-project/blob/main/hdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# HDB Resale Prices - Capstone Notebook

(Prepared by Antony Choi)

**Problem statement:** HDB resale prices vary widely across towns, flat types and lease ages. Buyers, sellers and policymakers find it hard to judge fair value. This notebook explores HDB resale transactions to find what drives resale prices and how the market has changed over time.

## Contents
1. Load the data
2. Understand the data
3. Exploratory data analysis
4. Clean the data
5. Prepare the data for Power BI


SyntaxError: invalid syntax. Perhaps you forgot a comma? (929373291.py, line 3)

In [3]:
## 1. Load the data
The 4 CSV files are loaded straight from this GitHub repository. The 2000 to 2012 file is read straight from its zip file.


SyntaxError: invalid syntax (2092656585.py, line 2)

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [6]:
base = 'https://raw.githubusercontent.com/antonytfchoi-glitch/capstone-project/main/'

df1 = pd.read_csv(base + 'resaleflat.zip')
df2 = pd.read_csv(base + 'resale-flat-prices-based-on-registration-date-from-mar-2012-to-dec-2014.csv')
df3 = pd.read_csv(base + 'resale-flat-prices-based-on-registration-date-from-jan-2015-to-dec-2016.csv')
df4 = pd.read_csv(base + 'ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv')

print('2000 to Feb 2012:', df1.shape)
print('Mar 2012 to 2014:', df2.shape)
print('2015 to 2016:', df3.shape)
print('2017 onwards:', df4.shape)


2000 to Feb 2012: (369651, 10)
Mar 2012 to 2014: (52203, 10)
2015 to 2016: (37153, 11)
2017 onwards: (169584, 11)


In [7]:
df = pd.concat([df1, df2, df3, df4], ignore_index=True)
print('Combined:', df.shape)


Combined: (628591, 11)



## 2. Understand the data
Check the combined table for its structure, missing values, duplicates and odd values. All problems found are listed in the data issues log at the end of this section.


In [8]:
df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,2000-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,NaN
1,2000-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,NaN
2,2000-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,NaN
3,2000-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,NaN
4,2000-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,NaN


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 628591 entries, 0 to 628590
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                628591 non-null  object 
 1   town                 628591 non-null  object 
 2   flat_type            628591 non-null  object 
 3   block                628591 non-null  object 
 4   street_name          628591 non-null  object 
 5   storey_range         628591 non-null  object 
 6   floor_area_sqm       628591 non-null  float64
 7   flat_model           628591 non-null  object 
 8   lease_commence_date  628591 non-null  int64  
 9   resale_price         628591 non-null  float64
 10  remaining_lease      206737 non-null  object 
dtypes: float64(2), int64(1), object(8)
memory usage: 52.8+ MB


In [10]:
df.describe()

,floor_area_sqm,lease_commence_date,resale_price
count,628591.000000,628591.000000,6.285910e+05
mean,96.767132,1990.411314,3.620510e+05
std,25.051891,11.335256,1.643300e+05
min,28.000000,1966.000000,2.800000e+04
25%,74.000000,1983.000000,2.400000e+05
50%,96.000000,1988.000000,3.380000e+05
75%,113.000000,1998.000000,4.500000e+05
max,297.000000,2022.000000,1.500000e+06


In [11]:
df.isnull().sum()

,0
month,0
town,0
flat_type,0
block,0
street_name,0
storey_range,0
floor_area_sqm,0
flat_model,0
lease_commence_date,0
resale_price,0


In [12]:
print('Exact duplicate rows:', df.duplicated().sum())
print('First month:', df['month'].min())
print('Last month:', df['month'].max())
print('Rows in 2024:', (df['month'].str[:4] == '2024').sum())


Exact duplicate rows: 1066
First month: 2000-01
Last month: 2024-01
Rows in 2024: 411


In [13]:
df['flat_type'].value_counts()

,count
flat_type,
4 ROOM,248960
3 ROOM,180081
5 ROOM,144616
EXECUTIVE,46835
2 ROOM,7366
1 ROOM,473
MULTI-GENERATION,260


In [14]:
print('Number of towns:', df['town'].nunique())
df['storey_range'].value_counts().sort_index()


Number of towns: 26


,count
storey_range,
01 TO 03,123425
01 TO 05,2700
04 TO 06,154522
06 TO 10,2474
07 TO 09,138379
10 TO 12,118344
11 TO 15,1259
13 TO 15,47219
16 TO 18,19283


In [15]:
print(df3['remaining_lease'].head(3))
print(df4['remaining_lease'].head(3))


0    70
1    65
2    64
Name: remaining_lease, dtype: int64
0    61 years 04 months
1    60 years 07 months
2    62 years 05 months
Name: remaining_lease, dtype: object


In [16]:
print('Prices under 60,000:', (df['resale_price'] < 60000).sum())
print('Highest price:', df['resale_price'].max())

for flat_type, group in df.groupby('flat_type'):
    q1 = group['resale_price'].quantile(0.25)
    q3 = group['resale_price'].quantile(0.75)
    iqr = q3 - q1
    outliers = ((group['resale_price'] < q1 - 1.5 * iqr) | (group['resale_price'] > q3 + 1.5 * iqr)).sum()
    print(flat_type, 'outliers:', outliers)


Prices under 60,000: 199
Highest price: 1500000.0
1 ROOM outliers: 0
2 ROOM outliers: 10
3 ROOM outliers: 1738
4 ROOM outliers: 5774
5 ROOM outliers: 4358
EXECUTIVE outliers: 707
MULTI-GENERATION outliers: 0


In [17]:
### Data issues log

| No. | Issue | Rows | Plan |
| --- | --- | --- | --- |
| 1 | Data comes in 4 files with different columns | All | Combine with `pd.concat` (done in section 1) |
| 2 | `remaining_lease` is missing before 2015 | 421,854 | Work out `remaining_lease_years` for every row from `lease_commence_date` |
| 3 | `remaining_lease` is a number in 2015 to 2016 and text from 2017 | 206,737 | Replace with `remaining_lease_years` and drop the original |
| 4 | `month` is text | All | Convert to a date. Add `year` and `month_num` |
| 5 | Exact duplicate rows | 1,066 | Flag with `is_dup` and keep. They may be real separate sales |
| 6 | 2 storey band systems (3 floors and 5 floors) | 6,838 in 5-floor bands | Add `storey_mid`, the middle floor of each band |
| 7 | 2024 has only 1 month of data | 411 | Remove from the clean data |
| 8 | Price outliers by the IQR rule | 12,587 | Keep. They are real sales. Use the median |
| 9 | Prices under 60,000 | 199 | Keep. Almost all are 1 ROOM flats in the early 2000s |
| 10 | Prices are not adjusted for inflation | All | Compare features within 1 year (2023) |
| 11 | Dictionary lists lat/lon columns that are not in the data | - | Use town names for the Power BI map |
| 12 | Dictionary says `story_range` but the data has `storey_range` | - | Use `storey_range` |


SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3979085855.py, line 9)